# DFU Repair-7 v1.6 — Preserve 38 Good Trials
Direct self-contained Colab notebook. No nested notebook loader, no Base64 sidecar, no notebook SHA wrapper. Uses embedded hex+gzip evidence from the completed V4 export and only trains the original seven incompatible fold-1 trials.


In [ ]:
# DFU REPAIR-7 v1.6 DIRECT SELF-CONTAINED RUNNER
# - No nested notebook loader
# - No Base64 transport
# - No notebook SHA wrapper
# - Existing 38 good trials remain read-only
# - MobileNetV3 seed=2028 fold=1 is restored from embedded V4 evidence without training
# - Only the original 7 incompatible fold-1 trials may call train_trial()

import os, sys, json, time, shutil, tarfile, urllib.request, importlib, subprocess, pickle, hashlib, zipfile, gzip, io
from pathlib import Path

import numpy as np
import pandas as pd

PINNED_CODE_COMMIT = "4472d66a4f918d42ce9cfdfb57ed6dd95bdb0f11"
ALGORITHM_SOURCE_COMMIT = "349143b4d8b16f885adce3559542f6c202a2bca1"
RUN_ID = "RELIABLE_DFU_CV_V3_MISSING38"
DRIVE_ROOT = Path("/content/drive/MyDrive/DFU-ImageGuard")
BACKUP_ROOT = Path("/content/drive/MyDrive/DFU-ImageGuard-Backup")
RUN_ROOT = DRIVE_ROOT / "runs" / RUN_ID
LOCKED_SPLIT = RUN_ROOT / "manifests" / "locked_outer_fold_assignments.csv"
QUARANTINE_ROOT = RUN_ROOT / "_legacy_quarantine_fold1_split_mismatch"
REPO_ROOT = Path("/content/DFU-ImageGuard-repair7-pinned")
ARCHIVE = Path("/content/DFU-ImageGuard-repair7-pinned.tar.gz")
EXTRACT_ROOT = Path("/content/DFU-ImageGuard-repair7-extract")
EXPORT_ZIP = Path("/content/DFU_REPAIR7_FINAL_EXPORT.zip")
AGG_PRED_PATH = RUN_ROOT / "tables" / "all_oof_predictions.csv"
AGG_METRIC_PATH = RUN_ROOT / "tables" / "fold_seed_metrics.csv"
RECOVER_GOOD_FROM_AGGREGATE = {("mobilenetv3_large", 2028, 0)}

MODELS = ("convnextv2_tiny", "mobilenetv3_large", "densenet121")
SEEDS = (2026, 2027, 2028)
FOLDS_ZERO = (0, 1, 2, 3, 4)
EXPECTED = [(m, s, f) for f in FOLDS_ZERO for s in SEEDS for m in MODELS]
REPAIR = {
    ("convnextv2_tiny", 2026, 0),
    ("convnextv2_tiny", 2027, 0),
    ("convnextv2_tiny", 2028, 0),
    ("densenet121", 2026, 0),
    ("densenet121", 2027, 0),
    ("mobilenetv3_large", 2026, 0),
    ("mobilenetv3_large", 2027, 0),
}
GOOD38 = [x for x in EXPECTED if x not in REPAIR]

if len(EXPECTED) != 45 or len(REPAIR) != 7 or len(GOOD38) != 38:
    raise RuntimeError("Repair protocol identity count is invalid.")

print("=" * 78)
print("DFU REPAIR-7 v1.6 ONLY")
print("38 good trials are read-only; only 7 incompatible fold-1 trials may train.")
print("=" * 78)

# Robust Drive mount
from google.colab import drive

def _drive_ready():
    return Path("/content/drive/MyDrive").is_dir() and RUN_ROOT.exists()

if not _drive_ready():
    last_err = None
    for attempt in range(1, 4):
        try:
            print(f"Google Drive mount attempt {attempt}/3 ...")
            if Path("/content/drive").exists():
                try:
                    drive.flush_and_unmount()
                except Exception:
                    pass
            drive.mount("/content/drive", force_remount=(attempt > 1))
            if Path("/content/drive/MyDrive").is_dir():
                last_err = None
                break
        except Exception as e:
            last_err = e
            print(f"mount attempt {attempt} failed: {type(e).__name__}: {e}")
            time.sleep(attempt)
    if last_err is not None or not Path("/content/drive/MyDrive").is_dir():
        raise RuntimeError(f"Google Drive mount failed after retries: {last_err}")

print("Google Drive mount: PASS")
if not RUN_ROOT.is_dir():
    raise RuntimeError(f"Expected existing V4/V3 run folder not found: {RUN_ROOT}")
if not LOCKED_SPLIT.is_file():
    raise RuntimeError(f"Locked split manifest not found: {LOCKED_SPLIT}")
sentinel = RUN_ROOT / "REPAIR7_V1_6_SENTINEL.txt"
tok = str(time.time_ns())
sentinel.write_text(tok, encoding="utf-8")
if sentinel.read_text(encoding="utf-8") != tok:
    raise RuntimeError("Drive read/write verification failed.")
print("Drive verification: PASS")

import torch
if not torch.cuda.is_available():
    raise RuntimeError("GPU runtime required. Colab: Runtime > Change runtime type > T4 GPU.")
print("GPU:", torch.cuda.get_device_name(0))

def sha256_file(path: Path, chunk=8 * 1024 * 1024):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

def atomic_json(path: Path, obj):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(obj, indent=2, default=str), encoding="utf-8")
    os.replace(tmp, path)

def atomic_csv(path: Path, frame: pd.DataFrame):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    frame.to_csv(tmp, index=False)
    os.replace(tmp, path)

def trial_path(model, seed, fold_zero):
    return RUN_ROOT / "trials" / model / f"seed_{seed}" / f"fold_{fold_zero + 1}"

locked = pd.read_csv(LOCKED_SPLIT)
required_split_cols = {"image_id", "group_id", "label", "relative_path", "outer_fold"}
if not required_split_cols.issubset(locked.columns):
    raise RuntimeError(f"Locked split is missing required columns: {required_split_cols - set(locked.columns)}")
if len(locked) != 1055 or locked.image_id.astype(str).duplicated().any():
    raise RuntimeError(f"Locked split expected 1055 unique images; got rows={len(locked)}, unique={locked.image_id.astype(str).nunique()}")
expected_by_fold = {
    f: set(locked.loc[locked.outer_fold.astype(int) == f, "image_id"].astype(str))
    for f in FOLDS_ZERO
}
print("Locked fold sizes:", {f + 1: len(v) for f, v in expected_by_fold.items()})
if sum(len(v) for v in expected_by_fold.values()) != 1055:
    raise RuntimeError("Locked fold partition does not cover exactly 1055 images.")

def validate_completed_identity(model, seed, fold_zero, require_current_split=True):
    t = trial_path(model, seed, fold_zero)
    cp = t / "COMPLETE.json"
    pp = t / "test_predictions.csv"
    if not (cp.is_file() and pp.is_file()):
        return {"valid": False, "reason": "missing_complete_or_predictions", "trial": str(t)}
    try:
        metrics = json.loads(cp.read_text(encoding="utf-8"))
        pred = pd.read_csv(pp)
        ef = fold_zero + 1
        if str(metrics.get("model_key")) != model or int(metrics.get("seed")) != seed or int(metrics.get("outer_fold")) != ef:
            return {"valid": False, "reason": "metrics_identity_mismatch", "trial": str(t)}
        if pred.empty:
            return {"valid": False, "reason": "empty_predictions", "trial": str(t)}
        need = {"image_id", "group_id", "label", "model_key", "seed", "outer_fold", "prob_calibrated", "pred"}
        if not need.issubset(pred.columns):
            return {"valid": False, "reason": "prediction_columns_missing", "trial": str(t)}
        ids = pred[["model_key", "seed", "outer_fold"]].drop_duplicates()
        if len(ids) != 1:
            return {"valid": False, "reason": "prediction_identity_not_unique", "trial": str(t)}
        row = ids.iloc[0]
        if str(row.model_key) != model or int(row.seed) != seed or int(row.outer_fold) != ef:
            return {"valid": False, "reason": "prediction_identity_mismatch", "trial": str(t)}
        got = set(pred.image_id.astype(str))
        if require_current_split:
            exp = expected_by_fold[fold_zero]
            if len(pred) != len(exp) or pred.image_id.astype(str).duplicated().any() or got != exp:
                return {"valid": False, "reason": "locked_split_mismatch", "rows": len(pred), "unique": len(got), "expected": len(exp), "unexpected": len(got-exp), "missing": len(exp-got), "trial": str(t)}
        return {"valid": True, "metrics": metrics, "pred": pred, "complete_sha": sha256_file(cp), "pred_sha": sha256_file(pp), "trial": str(t)}
    except Exception as e:
        return {"valid": False, "reason": f"exception:{type(e).__name__}:{e}", "trial": str(t)}

def _python_scalar(value):
    if pd.isna(value):
        return None
    if isinstance(value, np.generic):
        return value.item()
    return value

def recover_good_identity_from_aggregate(model, seed, fold_zero):
    identity = (model, seed, fold_zero)
    if identity not in RECOVER_GOOD_FROM_AGGREGATE:
        raise RuntimeError(f"Pinned export reconstruction is not authorized for {identity}")
    already = validate_completed_identity(model, seed, fold_zero, require_current_split=True)
    if already.get("valid"):
        print(f"Pinned V4 export recovery not needed: {model} seed={seed} fold={fold_zero+1}")
        return "already_present"
    ef = fold_zero + 1
    if identity != ("mobilenetv3_large", 2028, 0):
        raise RuntimeError(f"No pinned V4 export evidence is embedded for {identity}")
    _pred_blob = gzip.decompress(bytes.fromhex('1f8b0800a984776a02ffc57dcb6e654772eddc5fd1431ba04e6744e423727861c3f78e0c4f3c16f229e9bad412aaab6df4df7bc5399b2ab264746612a6590d554b87ac6264ee78ac15affdd3cfe587f1fd4ffde987cfbffce557fb974fa58e4f8fdfbfff53f9793c7d1e9fca979ffe637cff6bf9f2e3d3cfbf747ce1dfc75f9ffe3c467ffae52f5fc6e7efe72f9ff0077ff9e1a72f4fbf7efea57effb9fce7e35f5af9f453fd5cbee03b7ffd8cdfbe8c9f7f1df8efbf7c1e4f5f7efc3cfefc23fee4df39aaa4a38da481b3a4e9d3d4a77ffabfce89d2133dfdd33fff9bfdf3c77f2d5fda8fe3cf7ffc3ff54fbf7cfeb97cfafb7ffbd4c6e77ff8a34fe1f6ff7ffd01a2d59f3e8d3f8d2fff21df7f2a9f7f184fec58f11770bc69f444f9c9ddf2d75f497dd4fb479a39fa94323b4d12f127e49639e7c09e92cf4429257c5f4ace11be837cc007a47fe782b821d59386e1657a4e35dee50ec93fb9a77...'))
    _metric_blob = gzip.decompress(bytes.fromhex('1f8b0800a984776a02ff6553d18e9c3a0c7def57acfa322fd62a719c38e9cfa024385d5a064690696f7575fffd1aa6ddddb64208737c38b68fc302b9d6fb96eb0f2879ce4b957178457659f6a94fdfa6aef14dead4a67ac4b74de37d5a176816aeb5c2b6d621dfab26ce47...'))
    _pred_sha = hashlib.sha256(_pred_blob).hexdigest()
    _metric_sha = hashlib.sha256(_metric_blob).hexdigest()
    if _pred_sha != '0b931520810c3aa5c3c5bcc7abff91f2c8b0c1e6aa1a739cdf2ebef3ae0886a9':
        raise RuntimeError(f"Pinned V4 prediction evidence SHA mismatch: {_pred_sha}")
    if _metric_sha != '42e183909a73db5ab1e8ae1dcf09bc9ea4b7a9507fbd23594cf160066f01a766':
        raise RuntimeError(f"Pinned V4 metric evidence SHA mismatch: {_metric_sha}")
    p = pd.read_csv(io.BytesIO(_pred_blob))
    m = pd.read_csv(io.BytesIO(_metric_blob))
    expected_ids = expected_by_fold[fold_zero]
    got_ids = set(p.image_id.astype(str))
    if len(p) != len(expected_ids) or p.image_id.astype(str).duplicated().any() or got_ids != expected_ids:
        raise RuntimeError(f"Pinned V4 export evidence for {identity} does not exactly match locked fold {ef}: rows={len(p)}, unique={len(got_ids)}, expected={len(expected_ids)}, unexpected={len(got_ids-expected_ids)}, missing={len(expected_ids-got_ids)}")
    if len(m) != 1:
        raise RuntimeError(f"Expected exactly one pinned V4 metric row for {identity}, found {len(m)}")
    metric_record = {k: _python_scalar(v) for k, v in m.iloc[0].to_dict().items()}
    metric_record["model_key"] = model
    metric_record["seed"] = int(seed)
    metric_record["outer_fold"] = int(ef)
    t = trial_path(model, seed, fold_zero)
    t.mkdir(parents=True, exist_ok=True)
    backup_dir = RUN_ROOT / "_recovered_good_identity_backup" / model / f"seed_{seed}" / f"fold_{ef}"
    backup_dir.mkdir(parents=True, exist_ok=True)
    for name in ["COMPLETE.json", "test_predictions.csv"]:
        src = t / name
        if src.exists():
            shutil.copy2(src, backup_dir / f"{name}.{time.time_ns()}.bak")
    atomic_csv(t / "test_predictions.csv", p)
    atomic_json(t / "COMPLETE.json", metric_record)
    atomic_json(t / "RECOVERED_FROM_PINNED_V4_EXPORT.json", {"status": "PASS", "identity": [model, seed, ef], "prediction_rows": len(p), "prediction_sha256": _pred_sha, "metric_sha256": _metric_sha, "recovered_at_ns": time.time_ns(), "training_performed": False})
    verified = validate_completed_identity(model, seed, fold_zero, require_current_split=True)
    if not verified.get("valid"):
        raise RuntimeError(f"Pinned V4 export reconstruction failed verification for {identity}: {verified}")
    print(f"RECOVERED FROM PINNED V4 EXPORT (NO TRAINING): {model} seed={seed} fold={ef} | rows={len(p)}")
    return "recovered"

for _m, _s, _f in sorted(RECOVER_GOOD_FROM_AGGREGATE):
    recover_good_identity_from_aggregate(_m, _s, _f)

print("\nVerifying 38 good trials against the locked V4 split (including pinned-export-recovered evidence)...")
good38_before = {}
for i, (m, s, f) in enumerate(GOOD38, 1):
    chk = validate_completed_identity(m, s, f, require_current_split=True)
    if not chk.get("valid"):
        raise RuntimeError(f"Good-trial preflight failed for {(m,s,f+1)}: {chk}")
    good38_before[(m, s, f)] = (chk["complete_sha"], chk["pred_sha"])
print("38-good-trial preflight: PASS")

# Dependencies
required = [("timm","timm"),("kagglehub","kagglehub"),("imagehash","ImageHash"),("sklearn","scikit-learn"),("scipy","scipy"),("matplotlib","matplotlib"),("pandas","pandas"),("PIL","Pillow"),("tabulate","tabulate")]
for module_name, package_name in required:
    try:
        importlib.import_module(module_name)
    except Exception:
        p = subprocess.run([sys.executable,"-m","pip","install","--no-input","--disable-pip-version-check",package_name],text=True,capture_output=True)
        if p.returncode != 0:
            print(p.stdout[-2000:]); print(p.stderr[-4000:])
            raise RuntimeError(f"Dependency install failed: {package_name}")
        importlib.invalidate_caches(); importlib.import_module(module_name)
print("Dependency verification: PASS")

# Download exact pinned source archive; no git clone.
shutil.rmtree(REPO_ROOT, ignore_errors=True)
shutil.rmtree(EXTRACT_ROOT, ignore_errors=True)
ARCHIVE.unlink(missing_ok=True)
EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)
archive_url = "https://codeload.github.com/AzizulHakim00/DFU-ImageGuard/tar.gz/" + PINNED_CODE_COMMIT
last_download_error = None
for attempt in range(1,4):
    try:
        with urllib.request.urlopen(archive_url, timeout=120) as response, ARCHIVE.open("wb") as out:
            shutil.copyfileobj(response,out,length=8*1024*1024)
        if ARCHIVE.stat().st_size < 1024:
            raise RuntimeError("Pinned source archive unexpectedly small")
        last_download_error = None; break
    except Exception as e:
        last_download_error=e; print(f"source download attempt {attempt}/3 failed: {type(e).__name__}: {e}"); time.sleep(2*attempt)
if last_download_error is not None:
    raise RuntimeError(f"Pinned source download failed: {last_download_error}")
with tarfile.open(ARCHIVE,"r:gz") as tf:
    root=EXTRACT_ROOT.resolve()
    for member in tf.getmembers():
        target=(EXTRACT_ROOT/member.name).resolve()
        if target != root and root not in target.parents:
            raise RuntimeError(f"Unsafe archive member: {member.name}")
    tf.extractall(EXTRACT_ROOT)
folders=[p for p in EXTRACT_ROOT.iterdir() if p.is_dir()]
if len(folders)!=1:
    raise RuntimeError(f"Unexpected source archive layout: {folders}")
shutil.move(str(folders[0]),str(REPO_ROOT))
ARCHIVE.unlink(missing_ok=True); shutil.rmtree(EXTRACT_ROOT,ignore_errors=True)
sys.path.insert(0,str(REPO_ROOT))
print("Pinned source preparation: PASS")

from src import reliable_runner_v2 as rr
from src.reliable_storage_rescue import storage_bounded_torch_save, metadata_only_active_backup
from src.reliable_analysis import build_reports
rr.atomic_torch = storage_bounded_torch_save
rr.backup_active_trial = metadata_only_active_backup
settings = rr.ReliableSettingsV2(run_id=RUN_ID, drive_root=str(DRIVE_ROOT), backup_root=str(BACKUP_ROOT), source_commit=ALGORITHM_SOURCE_COMMIT)
cfg = rr.build_config(settings)

# Use locked split as the only split source. Dataset manifest is rebuilt only to obtain current image paths, then merged by immutable image_id.
dirs={"root":RUN_ROOT, **{name:RUN_ROOT/name for name in ("tables","figures","models","xai","predictions","logs","configs","manifests","cache")}}
for p in dirs.values(): Path(p).mkdir(parents=True,exist_ok=True)
dataset_root=rr.download_dataset(cfg,dirs)
current_manifest=rr.build_manifest(dataset_root,cfg,dirs)
current_manifest=current_manifest.loc[~current_manifest.exclude].copy()
path_map=current_manifest[["image_id","image_path"]].drop_duplicates("image_id")
data=locked.drop(columns=["image_path"],errors="ignore").merge(path_map,on="image_id",how="left",validate="one_to_one")
if data.image_path.isna().any():
    raise RuntimeError(f"Current dataset is missing {int(data.image_path.isna().sum())} locked images; refusing repair.")
print("Locked split + dataset path reconstruction: PASS")

def quarantine_bad_identity(model, seed, fold_zero):
    t=trial_path(model,seed,fold_zero)
    if not t.exists(): return
    q=QUARANTINE_ROOT/model/f"seed_{seed}"/f"fold_{fold_zero+1}_{time.time_ns()}"
    q.parent.mkdir(parents=True,exist_ok=True)
    shutil.move(str(t),str(q))
    print(f"QUARANTINED legacy incompatible trial: {model} seed={seed} fold={fold_zero+1}")

# Repair exact seven only.
for fold in FOLDS_ZERO:
    if fold != 0:
        continue
    outer_train=data[data.outer_fold.astype(int)!=fold].copy()
    test_df=data[data.outer_fold.astype(int)==fold].copy().reset_index(drop=True)
    inner=rr.make_inner_partition(outer_train,cfg,fold)
    train_df=inner[inner.inner_role=="train"].copy()
    selection_df=inner[inner.inner_role=="selection"].copy()
    calibration_df=inner[inner.inner_role=="calibration"].copy()
    for model_key, seed, repair_fold in sorted(REPAIR):
        if repair_fold != fold:
            continue
        existing=validate_completed_identity(model_key,seed,fold,require_current_split=True)
        if existing.get("valid"):
            print(f"SKIP already repaired: {model_key} seed={seed} fold=1")
            continue
        quarantine_bad_identity(model_key,seed,fold)
        t=trial_path(model_key,seed,fold)
        t.mkdir(parents=True,exist_ok=True)
        print("\n"+"-"*72)
        print(f"TRAIN REPAIR ONLY: {model_key} seed={seed} fold=1")
        print("-"*72)
        rr.train_trial(train_df=train_df,selection_df=selection_df,calibration_df=calibration_df,test_df=test_df,model_key=model_key,seed=seed,fold=fold,cfg=cfg,settings=settings,trial=t,run=RUN_ROOT)
        verified=validate_completed_identity(model_key,seed,fold,require_current_split=True)
        if not verified.get("valid"):
            raise RuntimeError(f"Repaired trial failed locked-split verification: {(model_key,seed,1)} -> {verified}")
        print(f"REPAIR COMPLETE: {model_key} seed={seed} fold=1")

# Final verify all 45 and prove 38 good evidence remained byte-identical.
metric_rows=[]; pred_frames=[]
for m,s,f in EXPECTED:
    chk=validate_completed_identity(m,s,f,require_current_split=True)
    if not chk.get("valid"):
        raise RuntimeError(f"Final trial verification failed for {(m,s,f+1)}: {chk}")
    metric_rows.append(chk["metrics"]); pred_frames.append(chk["pred"])
for identity,before in good38_before.items():
    chk=validate_completed_identity(*identity,require_current_split=True)
    after=(chk["complete_sha"],chk["pred_sha"])
    if after != before:
        raise RuntimeError(f"Protected good trial changed unexpectedly: {identity}; before={before}, after={after}")
print("38-good-trial post-run integrity: PASS")
metrics_df=pd.DataFrame(metric_rows).sort_values(["outer_fold","seed","model_key"]).reset_index(drop=True)
preds_df=pd.concat(pred_frames,ignore_index=True).sort_values(["outer_fold","seed","model_key","image_id"]).reset_index(drop=True)
if len(metrics_df)!=45:
    raise RuntimeError(f"Expected 45 metric rows, got {len(metrics_df)}")
if len(preds_df)!=9495:
    raise RuntimeError(f"Expected 9495 OOF rows, got {len(preds_df)}")
for (m,s),g in preds_df.groupby(["model_key","seed"]):
    if len(g)!=1055 or g.image_id.astype(str).nunique()!=1055:
        raise RuntimeError(f"OOF coverage failure for {(m,s)}: rows={len(g)}, unique={g.image_id.astype(str).nunique()}")
    exp_map=locked.set_index("image_id")["outer_fold"].astype(int)
    wrong=(g.set_index("image_id")["outer_fold"].astype(int)-exp_map.loc[g.image_id].to_numpy()).abs().sum()
    if wrong!=0:
        raise RuntimeError(f"Wrong fold assignments found for {(m,s)}")
atomic_csv(AGG_METRIC_PATH,metrics_df); atomic_csv(AGG_PRED_PATH,preds_df)
summary=metrics_df.groupby("model_key").agg({"balanced_accuracy":["mean","std"],"sensitivity":["mean","std"],"specificity":["mean","std"],"roc_auc":["mean","std"],"pr_auc":["mean","std"],"brier":["mean","std"],"ece":["mean","std"]})
summary.to_csv(RUN_ROOT/"tables"/"model_summary.csv")
report_status="PASS"
report_error=None
try:
    reports=build_reports(RUN_ROOT)
except Exception as e:
    report_status="DEGRADED"; report_error=f"{type(e).__name__}: {e}"; reports={}
    print("Report regeneration warning:",report_error)
with (RUN_ROOT/"reliable_dfu_reproducibility_repair7.pkl").open("wb") as handle:
    pickle.dump({"settings":settings.__dict__,"metrics":metrics_df.to_dict("records"),"predictions":preds_df.to_dict("records"),"reports":reports,"repair_version":"v1.6"},handle,pickle.HIGHEST_PROTOCOL)
final={"status":"PASS","repair_version":"v1.6","completed_unique_trials":45,"prediction_rows":len(preds_df),"good_trials_preserved":38,"repaired_trials":7,"unique_images_per_model_seed":1055,"locked_split_sha256":sha256_file(LOCKED_SPLIT),"report_status":report_status,"report_error":report_error,"completed_at_ns":time.time_ns()}
atomic_json(RUN_ROOT/"REPAIR7_FINAL_VERIFICATION.json",final)

# Compact final export
EXPORT_ZIP.unlink(missing_ok=True)
export_files=[AGG_METRIC_PATH,AGG_PRED_PATH,RUN_ROOT/"tables"/"model_summary.csv",RUN_ROOT/"tables"/"paired_bootstrap.json",RUN_ROOT/"tables"/"selective_prediction.csv",RUN_ROOT/"tables"/"error_audit.csv",RUN_ROOT/"reliable_dfu_reproducibility_repair7.pkl",RUN_ROOT/"REPAIR7_FINAL_VERIFICATION.json",LOCKED_SPLIT]
with zipfile.ZipFile(EXPORT_ZIP,"w",zipfile.ZIP_DEFLATED) as zf:
    for path in export_files:
        if path.is_file(): zf.write(path,arcname=str(path.relative_to(RUN_ROOT)))
print("\nREPAIR-7 FINAL VERIFICATION: PASS")
print(json.dumps(final,indent=2))
print("Final export:",EXPORT_ZIP)
try:
    from google.colab import files
    files.download(str(EXPORT_ZIP))
except Exception as e:
    print("Automatic export download skipped:",e)
